## Notebook 1: Corpus Construction and Data Acquisition (F0 → F2)

In [12]:
import sqlite3
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import re
import requests
import time

In [13]:
load_dotenv()
db_path = os.getenv('db_path')
conn = sqlite3.connect(db_path)

cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

tables

[('catalog',)]

In [14]:
catalog = pd.read_sql('SELECT * FROM catalog', conn)

print(catalog.shape)
catalog.head()

(61575, 10)


,gid,agents,bookshelves,creators,formats,languages,rights,subjects,title,types
0,0,NONE GIVEN,NONE GIVEN,NONE GIVEN,NONE GIVEN,EN,NONE,NONE GIVEN,NONE GIVEN,TEXT
1,1,NONE GIVEN,AMERICAN REVOLUTIONARY WAR\nPOLITICS\nUNITED S...,"JEFFERSON, THOMAS",APPLICATION/EPUB+ZIP\nAPPLICATION/EPUB+ZIP\nAP...,EN,PUBLIC DOMAIN IN THE USA.,UNITED STATES. DECLARATION OF INDEPENDENCE\nJK...,THE DECLARATION OF INDEPENDENCE OF THE UNITED ...,TEXT
2,2,NONE GIVEN,AMERICAN REVOLUTIONARY WAR\nUNITED STATES LAW\...,UNITED STATES,APPLICATION/EPUB+ZIP\nAPPLICATION/EPUB+ZIP\nAP...,EN,PUBLIC DOMAIN IN THE USA.,KF\nCIVIL RIGHTS -- UNITED STATES -- SOURCES\n...,THE UNITED STATES BILL OF RIGHTS THE TEN ORIGI...,TEXT
3,3,NONE GIVEN,NONE GIVEN,"KENNEDY, JOHN F. (JOHN FITZGERALD)",APPLICATION/EPUB+ZIP\nAPPLICATION/EPUB+ZIP\nAP...,EN,PUBLIC DOMAIN IN THE USA.,UNITED STATES -- FOREIGN RELATIONS -- 1961-196...,JOHN F. KENNEDY'S INAUGURAL ADDRESS,TEXT
4,4,NONE GIVEN,US CIVIL WAR,"LINCOLN, ABRAHAM",APPLICATION/EPUB+ZIP\nAPPLICATION/EPUB+ZIP\nAP...,EN,PUBLIC DOMAIN IN THE USA.,"SOLDIERS' NATIONAL CEMETERY (GETTYSBURG, PA.)\...",LINCOLN'S GETTYSBURG ADDRESS GIVEN NOVEMBER 19...,TEXT


In [15]:
scifi = catalog[
    catalog['bookshelves'].str.contains('SCIENCE FICTION', na=False) &
    catalog['languages'].str.contains('EN', na=False)
]

print(f"Total sci-fi books: {len(scifi)}")
print(scifi[['gid', 'title', 'creators']].to_string())

Total sci-fi books: 1358
         gid                                                                                                                   title                                                         creators
35        35                                                                                                        THE TIME MACHINE                                    WELLS, H. G. (HERBERT GEORGE)
36        36                                                                                                   THE WAR OF THE WORLDS                                    WELLS, H. G. (HERBERT GEORGE)
42        42                                                                             THE STRANGE CASE OF DR. JEKYLL AND MR. HYDE                                          STEVENSON, ROBERT LOUIS
43        43                                                                             THE STRANGE CASE OF DR. JEKYLL AND MR. HYDE                                          STEVENSON

In [16]:
authors_kept = ['PIPER, H. BEAM', 'VERNE, JULES', 'BURROUGHS, EDGAR RICE']

corpus = scifi[
    scifi['creators'].str.contains('|'.join(authors_kept), na=False)
].drop_duplicates(subset='title')  # remove duplicate titles

# Add a author column
def get_author(creators):
    for a in authors_kept:
        if a in creators:
            return a
    return 'OTHER'

corpus = corpus.copy()
corpus['author'] = corpus['creators'].apply(get_author)

print(corpus.groupby('author')['title'].count())
print(f"\nTotal books: {len(corpus)}")
print(corpus[['gid', 'title', 'author']])

author
BURROUGHS, EDGAR RICE    13
PIPER, H. BEAM           32
VERNE, JULES             15
Name: title, dtype: int64

Total books: 60
         gid                                              title  \
62        62                                 A PRINCESS OF MARS   
64        64                                   THE GODS OF MARS   
68        68                                    WARLORD OF MARS   
72        72                               THUVIA, MAID OF MARS   
83        83    FROM THE EARTH TO THE MOON; AND, ROUND THE MOON   
96        96                                    THE MONSTER MEN   
123      123                                AT THE EARTH'S CORE   
149      149                                 THE LOST CONTINENT   
164      164              TWENTY THOUSAND LEAGUES UNDER THE SEA   
369      369                                 THE OUTLAW OF TORN   
551      551                          THE LAND THAT TIME FORGOT   
552      552                        THE PEOPLE THAT TIME FORGO

In [17]:
duplicate_gids = [2488, 6538, 8983, 8984, 8986, 12901, 16457, 18857, 19513]
corpus = corpus[~corpus['gid'].isin(duplicate_gids)].reset_index(drop=True)

print(f"Final corpus size: {len(corpus)} books")
print(corpus.groupby('author')['title'].count())

Final corpus size: 51 books
author
BURROUGHS, EDGAR RICE    13
PIPER, H. BEAM           32
VERNE, JULES              6
Name: title, dtype: int64


In [18]:
for _, row in corpus.iterrows():
    gid = row["gid"]
    path = f"../data/RawTexts/{gid}.txt"
    
    if not os.path.exists(path):
        print(f"Downloading gid {gid}: {row['title'][:50]}")
        url = f"https://www.gutenberg.org/cache/epub/{gid}/pg{gid}.txt"

        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(path, "w", encoding="utf-8", errors="ignore") as f:
                    f.write(r.text)
                print("Downloaded")
            else:
                print(f"Failed: status code {r.status_code}")
            time.sleep(1)
        
        except Exception as e:
            print(f"Error downloading {gid}: {e}")
    
    else:
        print(f"Already have gid {gid}: {row['title'][:50]}")

Already have gid 62: A PRINCESS OF MARS
Already have gid 64: THE GODS OF MARS
Already have gid 68: WARLORD OF MARS
Already have gid 72: THUVIA, MAID OF MARS
Already have gid 83: FROM THE EARTH TO THE MOON; AND, ROUND THE MOON
Already have gid 96: THE MONSTER MEN
Already have gid 123: AT THE EARTH'S CORE
Already have gid 149: THE LOST CONTINENT
Already have gid 164: TWENTY THOUSAND LEAGUES UNDER THE SEA
Already have gid 369: THE OUTLAW OF TORN
Already have gid 551: THE LAND THAT TIME FORGOT
Already have gid 552: THE PEOPLE THAT TIME FORGOT
Already have gid 553: OUT OF TIME'S ABYSS
Already have gid 605: PELLUCIDAR
Already have gid 1153: THE CHESSMEN OF MARS
Already have gid 1268: THE MYSTERIOUS ISLAND
Already have gid 1353: OFF ON A COMET! A JOURNEY THROUGH PLANETARY SPACE
Already have gid 3748: A JOURNEY INTO THE INTERIOR OF THE EARTH
Already have gid 18105: GENESIS
Already have gid 18109: GRAVEYARD OF DREAMS
Already have gid 18137: LITTLE FUZZY
Already have gid 18151: TIME CRIME
Alread

In [19]:
def strip_gutenberg(text):
    start_markers = [
        r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK .+? \*\*\*",
        r"\*\*\* START OF THIS PROJECT GUTENBERG EBOOK .+? \*\*\*",
    ]
    
    for marker in start_markers:
        match = re.search(marker, text, re.IGNORECASE)
        if match:
            text = text[match.end():]
            break
    
    end_markers = [
        r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK .+? \*\*\*",
        r"\*\*\* END OF THIS PROJECT GUTENBERG EBOOK .+? \*\*\*",
    ]
    
    for marker in end_markers:
        match = re.search(marker, text, re.IGNORECASE)
        if match:
            text = text[:match.start()]
            break
    
    return text.strip()

In [20]:
texts = {}

for _, row in corpus.iterrows():
    gid = row["gid"]
    path = f"../data/RawTexts/{gid}.txt"
    
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    
    texts[gid] = strip_gutenberg(raw)

for gid, text in texts.items():
    clean_path = f"../data/CleanTexts/{gid}.txt"
    with open(clean_path, "w", encoding="utf-8") as f:
        f.write(text)
    title = corpus.loc[corpus["gid"] == gid, "title"].values[0]
    print(f"{gid} | {len(text):>10,} chars | {title[:50]}")

62 |    378,163 chars | A PRINCESS OF MARS
64 |    459,695 chars | THE GODS OF MARS
68 |    317,806 chars | WARLORD OF MARS
72 |    272,243 chars | THUVIA, MAID OF MARS
83 |    556,261 chars | FROM THE EARTH TO THE MOON; AND, ROUND THE MOON
96 |    326,397 chars | THE MONSTER MEN
123 |    273,488 chars | AT THE EARTH'S CORE
149 |    213,436 chars | THE LOST CONTINENT
164 |    610,261 chars | TWENTY THOUSAND LEAGUES UNDER THE SEA
369 |    362,876 chars | THE OUTLAW OF TORN
551 |    203,756 chars | THE LAND THAT TIME FORGOT
552 |    212,084 chars | THE PEOPLE THAT TIME FORGOT
553 |    204,739 chars | OUT OF TIME'S ABYSS
605 |    317,575 chars | PELLUCIDAR
1153 |    495,541 chars | THE CHESSMEN OF MARS
1268 |  1,135,368 chars | THE MYSTERIOUS ISLAND
1353 |    604,255 chars | OFF ON A COMET! A JOURNEY THROUGH PLANETARY SPACE
3748 |    428,743 chars | A JOURNEY INTO THE INTERIOR OF THE EARTH
18105 |     50,569 chars | GENESIS
18109 |     46,890 chars | GRAVEYARD OF DREAMS
18137 |    345,270

In [21]:
LIBRARY = corpus[["gid", "title", "author", "subjects", "bookshelves"]].copy()
LIBRARY = LIBRARY.rename(columns={"gid": "book_id"})

LIBRARY["genre"] = "science fiction"
LIBRARY["char_count"] = LIBRARY["book_id"].apply(lambda gid: len(texts[gid]))
LIBRARY["word_count"] = LIBRARY["book_id"].apply(lambda gid: len(texts[gid].split()))

LIBRARY = LIBRARY.set_index("book_id")
LIBRARY.head()

,title,author,subjects,bookshelves,genre,char_count,word_count
book_id,,,,,,,
62,A PRINCESS OF MARS,"BURROUGHS, EDGAR RICE",SCIENCE FICTION\nMARS (PLANET) -- FICTION\nDEJ...,SCIENCE FICTION\nBEST BOOKS EVER LISTINGS,science fiction,378163,67436
64,THE GODS OF MARS,"BURROUGHS, EDGAR RICE",PS\nLIFE ON OTHER PLANETS -- FICTION\nDEJAH TH...,SCIENCE FICTION,science fiction,459695,82741
68,WARLORD OF MARS,"BURROUGHS, EDGAR RICE","CARTER, JOHN (FICTITIOUS CHARACTER) -- FICTION...",SCIENCE FICTION,science fiction,317806,57134
72,"THUVIA, MAID OF MARS","BURROUGHS, EDGAR RICE",SCIENCE FICTION\nPS\nMARS (PLANET) -- FICTION\...,SCIENCE FICTION,science fiction,272243,47143
83,"FROM THE EARTH TO THE MOON; AND, ROUND THE MOON","VERNE, JULES",MOON -- FICTION\nPQ\nMANNED SPACE FLIGHT -- FI...,SCIENCE FICTION\nMOVIE BOOKS,science fiction,556261,91500


In [22]:
LIBRARY["source_url"] = LIBRARY.index.map(
    lambda gid: f"https://www.gutenberg.org/ebooks/{gid}"
)
LIBRARY.to_csv("../data/OutputTables/LIBRARY.csv")